# 002 · Creating Modules and Packages

> **Syntax › Modules & Packages** · Python 3.14

Writing your own module is as simple as saving a `.py` file; a **package** is a folder of modules. In this notebook we build a small package on disk (inside a temporary directory), import it, and explore how Python's packaging concepts line up with C# projects and namespaces.

**In this notebook you will learn:**

- how to turn a folder into a package with `__init__.py`
- absolute vs relative imports inside a package
- the `if __name__ == "__main__":` idiom (≈ `static void Main`)
- controlling the public surface with `__all__`
- regular vs namespace packages, and the recommended `src` layout
- how this compares to `.csproj` files and project references

## C# mental model

| C# | Python |
|---|---|
| `.cs` file | module (`.py` file) — and the file name **is** the namespace |
| namespace (folder convention) | package (folder with `__init__.py`) |
| project (`.csproj`) → assembly | project (`pyproject.toml`) → distribution (wheel) |
| `public` / `internal` | convention: `__all__` and `_leading_underscore` names |
| `static void Main(string[] args)` | `if __name__ == "__main__":` block |
| `<ProjectReference>` | dependency on a local package (path / workspace member) |

The key mental shift: in C#, namespaces are *declared* (`namespace Shop.Models;`) and are independent of file names. In Python, the **file system is the namespace**: `shop/models.py` *is* `shop.models`.

## Setting up a scratch area

We never write into the repository. Everything below lives in a `TemporaryDirectory` that is added to `sys.path` and removed at the end.

In [ ]:
import sys
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()
root = Path(tmp.name)
sys.path.insert(0, str(root))       # like adding a probing path


def write(relative: str, code: str) -> None:
    """Create a file (and any parent folders) under the scratch root."""
    path = root / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(code)


print("scratch root ready:", root.exists())

## A single module

A module is just a file. Its top-level functions, classes and variables become attributes of the module object.

In [ ]:
write("textutils.py", '''
"""Small text helpers."""

def slugify(text: str) -> str:
    return "-".join(text.lower().split())

def _normalise(text: str) -> str:      # leading underscore = "internal"
    return text.strip()
''')

import textutils

print(textutils.slugify("Hello Python World"))
print(textutils.__doc__)

## Packages and `__init__.py`

A **package** is a folder containing an `__init__.py` file. It groups related modules under one dotted name. Let's build this layout:

```text
shop/
├── __init__.py        # runs when "shop" is first imported
├── models.py          # shop.models
├── pricing.py         # shop.pricing
└── reports/
    ├── __init__.py
    └── summary.py     # shop.reports.summary
```

```csharp
// The C# equivalent would be folders + namespace declarations
namespace Shop.Models;     // Models/Product.cs
namespace Shop.Pricing;    // Pricing/PriceCalculator.cs
```

In [ ]:
write("shop/models.py", '''
from dataclasses import dataclass

@dataclass
class Product:
    name: str
    price: float
''')

write("shop/pricing.py", '''
from .models import Product           # relative import: "models in my package"

VAT_RATE = 0.2

def with_vat(product: Product) -> float:
    return round(product.price * (1 + VAT_RATE), 2)
''')

write("shop/reports/__init__.py", "")

write("shop/reports/summary.py", '''
from ..pricing import with_vat        # relative: go up one package level
from shop.models import Product       # absolute: full path from the top

def total(products: list[Product]) -> float:
    return sum(with_vat(p) for p in products)
''')

`__init__.py` runs the first time the package is imported. It's commonly used to re-export the most useful names so callers can write `from shop import Product` instead of `from shop.models import Product` — similar to a facade.

In [ ]:
write("shop/__init__.py", '''
"""A tiny demo shop package."""
print("  (shop/__init__.py running)")

from .models import Product
from .pricing import with_vat

__all__ = ["Product", "with_vat"]
__version__ = "1.0.0"
''')

import shop
from shop import Product, with_vat
from shop.reports.summary import total

items = [Product("widget", 10.0), Product("gadget", 25.0)]
print(with_vat(items[0]), total(items), shop.__version__)

## Absolute vs relative imports

Inside a package you can refer to sibling modules in two ways:

| Style | Example | Meaning |
|---|---|---|
| Absolute | `from shop.models import Product` | full path from a `sys.path` root |
| Relative | `from .models import Product` | `.` = this package |
| Relative | `from ..pricing import with_vat` | `..` = parent package |

Both are fine. Absolute imports are explicit and greppable (PEP 8 prefers them); relative imports make it easy to rename or move the whole package. Pick one style per project.

⚠️ **Gotcha:** relative imports only work in modules that are imported **as part of a package**. Running a package file directly as a script fails:

In [ ]:
import os
import subprocess

plain = {**os.environ, "PYTHON_COLORS": "0"}   # no ANSI colours in captured tracebacks (3.13+)
result = subprocess.run(
    [sys.executable, str(root / "shop" / "pricing.py")],   # run the file directly
    capture_output=True, text=True, env=plain,
)
print(result.stderr.strip().splitlines()[-1])

The fix is to run it as a module with `-m`, from the folder that contains the package: `python -m shop.pricing`. Then Python knows the package context.

## `if __name__ == "__main__":` — the entry point

Every module has a `__name__`. When imported it's the module's dotted name; when the file is **executed directly** (`python file.py` or `python -m package.module`) it is the string `"__main__"`. This lets one file be both an importable library and a runnable script.

```csharp
// C#
public static class Program
{
    public static void Main(string[] args) => Console.WriteLine("running as a program");
}
```

In [ ]:
write("shop/cli.py", '''
import sys
from shop import Product, with_vat

def main(argv: list[str]) -> int:
    name, price = argv[0], float(argv[1])
    print(f"{name}: {with_vat(Product(name, price))} incl. VAT")
    return 0

print(f"  cli.py loaded with __name__ = {__name__!r}")

if __name__ == "__main__":            # only when executed, not when imported
    raise SystemExit(main(sys.argv[1:]))
''')

import shop.cli                       # imported: the main() call is skipped

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "shop.cli", "laptop", "1000"],   # executed as the main module
    capture_output=True, text=True, cwd=root, env=plain,
)
print(result.stdout)

💡 Put the real work in a `main()` function and keep the `if __name__ == "__main__":` block to one line. That way the logic stays importable and testable.

## `__all__` — the public surface

Python has no `public`/`internal` keywords. Two conventions take their place:

- names starting with `_` are **internal** — nothing stops access, but tools and readers treat them as private
- `__all__` lists the names exported by `from module import *` and documents the intended public API

In [ ]:
write("geometry.py", '''
__all__ = ["area"]

def area(w, h):
    return w * h

def perimeter(w, h):        # public name, but not in __all__
    return 2 * (w + h)

def _validate(w, h):        # internal by convention
    return w > 0 and h > 0
''')

namespace: dict = {}
exec("from geometry import *", namespace)
print(sorted(k for k in namespace if not k.startswith("__")))   # only 'area'

import geometry
print(geometry.perimeter(2, 3), geometry._validate(2, 3))       # still reachable explicitly

## Regular vs namespace packages

A folder **without** `__init__.py` can still be imported: it becomes a **namespace package**. Its distinctive feature is that it can be split across several directories on `sys.path` — useful for plugin ecosystems where separately installed distributions contribute to one package name (somewhat like several assemblies contributing types to the same C# namespace).

In [ ]:
write("site_a/plugins/csv_export.py", "NAME = 'csv'\n")
write("site_b/plugins/pdf_export.py", "NAME = 'pdf'\n")
sys.path[:0] = [str(root / "site_a"), str(root / "site_b")]

import plugins.csv_export
import plugins.pdf_export

print(plugins.csv_export.NAME, plugins.pdf_export.NAME)
print(type(plugins.__path__).__name__, len(list(plugins.__path__)), "directories")

💡 For your own code, always add `__init__.py` (a regular package). Namespace packages are a specialised tool, and forgetting `__init__.py` by accident can produce confusing import behaviour.

## Project layout: the `src` layout

For a real, installable project the recommended structure is the **src layout**:

```text
my-shop/
├── pyproject.toml          # ≈ .csproj: metadata + dependencies
├── README.md
├── src/
│   └── shop/               # the importable package
│       ├── __init__.py
│       ├── models.py
│       └── pricing.py
└── tests/
    └── test_pricing.py
```

Putting the package under `src/` stops Python from importing it accidentally from the current directory — tests run against the **installed** package, just like a C# test project references the built assembly rather than raw source files. `uv init --lib` creates this layout for you *(see Modules & Packages › 003 · Environments and Dependencies with uv)*.

## Compared with `.csproj` and project references

| .NET | Python |
|---|---|
| `.sln` with several projects | a uv **workspace** (several packages, one lockfile) |
| `.csproj` | `pyproject.toml` |
| assembly name / root namespace | distribution name / top-level package name |
| `<ProjectReference Include="../Shop.Core/Shop.Core.csproj" />` | a path or workspace dependency, e.g. `uv add --editable ../shop-core` |
| `dotnet pack` → `.nupkg` | `uv build` → wheel (`.whl`) + source distribution |
| `InternalsVisibleTo` | not needed — nothing is truly private |

⚠️ **Gotcha:** the *distribution* name (what you install, e.g. `python-dateutil`) can differ from the *import* name (`dateutil`). It's like a NuGet package ID that differs from its root namespace.

## Cleaning up

Remove our modules from the import cache and `sys.path`, then delete the temporary directory:

In [ ]:
for name in list(sys.modules):
    if name.split(".")[0] in {"shop", "textutils", "geometry", "plugins"}:
        del sys.modules[name]

for entry in (root, root / "site_a", root / "site_b"):
    sys.path.remove(str(entry))

tmp.cleanup()
print("shop" in sys.modules, root.exists())

## Summary

| Concept | C# | Python |
|---|---|---|
| Code file | `.cs` | module `.py` (file name = module name) |
| Namespace | `namespace Shop.Models;` | package folder `shop/models.py` → `shop.models` |
| Package marker | — | `__init__.py` |
| Re-export / facade | — | imports in `__init__.py` |
| Same-package import | `using Shop.Models;` | `from .models import Product` |
| Entry point | `static void Main` | `if __name__ == "__main__":` |
| Run a program | `dotnet run` | `python -m package.module` |
| Public API | `public` / `internal` | `__all__` + `_private` convention |
| Project file | `.csproj` | `pyproject.toml` |
| Project reference | `<ProjectReference>` | path / workspace dependency |

## Exercises

1. Inside a `TemporaryDirectory`, create a package `mathx` with modules `basic.py` (`add`, `sub`) and `advanced.py` (`double(x)`, implemented by calling `add`, which it imports from `basic` with a **relative** import). Re-export `add` and `double` from `mathx/__init__.py`, then import and call them. Clean up afterwards.
2. Add `__all__ = ["add"]` to `basic.py` and verify with `exec("from mathx.basic import *", ns)` that `sub` is not exported.
3. What value does `__name__` have inside `mathx/advanced.py` when it's imported, and when it's run with `python -m mathx.advanced`?

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import sys
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as d:
    root = Path(d)
    (root / "mathx").mkdir()
    # 1 + 2
    (root / "mathx" / "basic.py").write_text(
        '__all__ = ["add"]\n'
        "def add(a, b): return a + b\n"
        "def sub(a, b): return a - b\n"
    )
    (root / "mathx" / "advanced.py").write_text(
        "from .basic import add\n"
        "def double(x):\n"
        "    return add(x, x)\n"
    )
    (root / "mathx" / "__init__.py").write_text(
        "from .basic import add\nfrom .advanced import double\n"
    )
    sys.path.insert(0, str(root))
    try:
        from mathx import add, double
        print(add(2, 3), double(21))     # 5 42

        ns = {}
        exec("from mathx.basic import *", ns)
        print("sub" in ns, "add" in ns)  # False True
    finally:
        sys.path.remove(str(root))
        for name in [m for m in sys.modules if m.split(".")[0] == "mathx"]:
            del sys.modules[name]

# 3 — imported: "mathx.advanced"; run with -m: "__main__"
```

</details>